# 💻 Laptop Price Prediction
### End-to-End ML Project | Real Kaggle Dataset | 1303 Laptops

> **Dataset**: Laptop Price by Muhammet Varlı — [Kaggle](https://www.kaggle.com/datasets/muhammetvarol/laptop-prices-dataset)  
> **Target**: `Price_euros` — predict laptop price in Euros  
> **Models**: Linear Regression, Ridge, Decision Tree, Random Forest, Gradient Boosting

| File | Purpose |
|------|---------|
| `src.ipynb` | Research & EDA (this file) |
| `project_root/training/train.py` | Production training script |
| `project_root/api/app.py` | Flask REST API |
| `Dockerfile` | Docker deployment |

---

## 📦 1. Imports

In [ ]:
import re, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('✅ Libraries loaded')

## 🗂️ 2. Load & Explore Raw Data

In [ ]:
df = pd.read_csv('project_root/data/laptops.csv', encoding='latin1')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['Price_euros'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Laptop Price Distribution')
axes[0].set_xlabel('Price (€)')

brand_avg = df.groupby('Company')['Price_euros'].mean().sort_values(ascending=False)
axes[1].bar(brand_avg.index, brand_avg.values, color='coral', edgecolor='white')
axes[1].set_title('Average Price by Brand')
axes[1].set_ylabel('Avg Price (€)')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Price by laptop type
plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='TypeName', y='Price_euros', palette='Set2')
plt.title('Price Distribution by Laptop Type')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## ⚙️ 3. Feature Engineering

In [ ]:
df2 = df.drop(columns=['laptop_ID','Product'], errors='ignore').copy()

# RAM: '8GB' → 8
df2['Ram_GB'] = df2['Ram'].str.replace('GB','').astype(int)
df2 = df2.drop(columns=['Ram'])

# Weight: '1.37kg' → 1.37
df2['Weight_KG'] = df2['Weight'].str.replace('kg','').astype(float)
df2 = df2.drop(columns=['Weight'])

# Memory → total GB + SSD flag
def total_storage(mem):
    total = 0
    for p in str(mem).replace('Hybrid','SSD').upper().split('+'):
        try:
            val = float(re.search(r'[\d\.]+', p).group())
            if 'TB' in p: val *= 1024
            total += val
        except: pass
    return total

df2['Storage_GB'] = df2['Memory'].apply(total_storage)
df2['SSD']        = df2['Memory'].apply(lambda x: 1 if 'SSD' in str(x).upper() or 'FLASH' in str(x).upper() else 0)
df2 = df2.drop(columns=['Memory'])

# CPU brand
df2['CPU_Brand'] = df2['Cpu'].apply(lambda x: 'Intel' if 'Intel' in x else ('AMD' if 'AMD' in x else 'Other'))
df2 = df2.drop(columns=['Cpu'])

# GPU brand
df2['GPU_Brand'] = df2['Gpu'].apply(lambda x: 'Nvidia' if 'Nvidia' in x else ('Intel' if 'Intel' in x else ('AMD' if 'AMD' in x else 'Other')))
df2 = df2.drop(columns=['Gpu'])

# Screen resolution features
def get_pixels(res):
    m = re.search(r'(\d{3,4})x(\d{3,4})', str(res))
    return int(m.group(1))*int(m.group(2)) if m else 1920*1080

df2['Resolution_MP'] = df2['ScreenResolution'].apply(get_pixels)
df2['IPS']           = df2['ScreenResolution'].apply(lambda x: 1 if 'IPS' in str(x) else 0)
df2['Touchscreen']   = df2['ScreenResolution'].apply(lambda x: 1 if 'Touch' in str(x) else 0)
df2 = df2.drop(columns=['ScreenResolution'])

# Simplify OS
def simplify_os(os):
    os = str(os).lower()
    if 'windows' in os: return 'Windows'
    if 'mac' in os: return 'macOS'
    if 'linux' in os: return 'Linux'
    if 'chrome' in os: return 'Chrome OS'
    return 'Other'
df2['OpSys'] = df2['OpSys'].apply(simplify_os)

print('✅ Features after engineering:')
print(df2.columns.tolist())
df2.head()

In [ ]:
# Correlation heatmap on numeric features
num_cols = ['Inches','Ram_GB','Weight_KG','Storage_GB','SSD','Resolution_MP','IPS','Touchscreen','Price_euros']
plt.figure(figsize=(10,7))
sns.heatmap(df2[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 🔢 4. Encoding & Train/Test Split

In [ ]:
df_enc = df2.copy()
label_encoders = {}
for col in ['Company','TypeName','OpSys','CPU_Brand','GPU_Brand']:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    label_encoders[col] = le
    print(f'{col}: {list(le.classes_)}')

X = df_enc.drop(columns=['Price_euros'])
y = df_enc['Price_euros']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 🤖 5. Train & Compare 5 Models

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=10),
    'Decision Tree'     : DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results.append({'Model':name,'MAE':round(mae,2),'RMSE':round(rmse,2),'R2':round(r2,4)})
    print(f'{name:<25} | R²={r2:.4f} | MAE=€{mae:.2f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Model comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14,5))
colors = ['gold' if i==0 else 'steelblue' for i in range(len(results_df))]
axes[0].barh(results_df['Model'], results_df['R2'], color=colors, edgecolor='white')
axes[0].set_title('R² Score (higher = better)'); axes[0].invert_yaxis()
axes[1].barh(results_df['Model'], results_df['MAE'], color='coral', edgecolor='white')
axes[1].set_title('MAE in € (lower = better)'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# Actual vs Predicted
best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred_best= best_model.predict(X_test)

plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', edgecolors='white', s=40)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--',lw=2,label='Perfect')
plt.xlabel('Actual Price (€)'); plt.ylabel('Predicted Price (€)')
plt.title(f'Actual vs Predicted — {best_name}')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
    imp.plot(kind='barh', color='mediumseagreen', edgecolor='white', figsize=(9,6))
    plt.title(f'Feature Importances — {best_name}')
    plt.tight_layout(); plt.show()

## 💾 6. Save Model

In [ ]:
import os
os.makedirs('project_root/models', exist_ok=True)

artifact = {
    'model': best_model, 'label_encoders': label_encoders,
    'feature_names': list(X.columns), 'model_name': best_name,
    'r2': results_df.iloc[0]['R2'], 'mae': results_df.iloc[0]['MAE']
}
with open('project_root/models/laptop_price_model.pkl','wb') as f:
    pickle.dump(artifact, f)
with open('columns_label_encodings.pkl','wb') as f:
    pickle.dump(label_encoders, f)

print(f'✅ Saved: {best_name} | R²={results_df.iloc[0]["R2"]} | MAE=€{results_df.iloc[0]["MAE"]}')

## 🔮 7. Predict a New Laptop

In [ ]:
def predict_price(company, type_name, inches, ram_gb, storage_gb, ssd,
                  weight_kg, op_sys, cpu_brand, gpu_brand,
                  resolution_mp=2073600, ips=1, touchscreen=0):
    with open('project_root/models/laptop_price_model.pkl','rb') as f:
        art = pickle.load(f)
    inp = {
        'Company':company, 'TypeName':type_name, 'Inches':inches,
        'Ram_GB':ram_gb, 'Storage_GB':storage_gb, 'SSD':ssd,
        'Weight_KG':weight_kg, 'OpSys':op_sys, 'CPU_Brand':cpu_brand,
        'GPU_Brand':gpu_brand, 'Resolution_MP':resolution_mp,
        'IPS':ips, 'Touchscreen':touchscreen
    }
    for col, le in art['label_encoders'].items():
        inp[col] = int(le.transform([inp[col]])[0])
    df_in = pd.DataFrame([inp])[art['feature_names']]
    price = round(float(art['model'].predict(df_in)[0]), 2)
    return price

# Example: Dell Notebook, 15.6", 8GB RAM, 256GB SSD, Windows
price = predict_price('Dell','Notebook',15.6,8,256,1,2.0,'Windows','Intel','Nvidia')
print(f'🎯 Predicted Price: €{price:,.2f}  (~${price*1.08:,.2f} USD)')